# 🧪 Eksperimen USD/IDR Forecasting — Kaggle Kernel
## Rupiah Resilience

**Penting:** Aktifkan **Internet** dan **GPU** sebelum run!

1. Settings → Accelerator → **GPU T4 x2** or **GPU P100**
2. Settings → Internet → **On**

Pipeline:
1. Hyperparameter Optimization (Optuna)
2. Training & Benchmarking (3 skenario × semua model)
3. Evaluasi & Metrik
4. Future Forecasting

## Cell 1 — Clone Repo & Install

In [ ]:
import os

# Clone repo
REPO_URL = "https://github.com/ruwwww/fp-pap.git"
REPO_NAME = "fp-pap"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

os.chdir(f"/kaggle/working/{REPO_NAME}")
print(f"Working dir: {os.getcwd()}")
!ls

In [ ]:
# Install dependencies (most pre-installed di Kaggle, tambah yg kurang)
!pip install -q optuna lightgbm pyyaml

import sys, platform
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## Cell 2 — Tahap 1a: Hyperparameter Search — ML Models

In [ ]:
!python scripts/search.py --category ML --trials 50 --jobs 4

## Cell 3 — Tahap 1b: Hyperparameter Search — DL Models

In [ ]:
!python scripts/search.py --category DL --trials 20

## Cell 4 — Tahap 2: Training & Benchmarking

In [ ]:
!python scripts/train.py --data data/raw/data_train.csv --use-best-params --parallel --workers 4

## Cell 5 — Tahap 3: Evaluasi

In [ ]:
!python scripts/evaluate.py --report-dir report/

## Cell 6 — Tahap 4: Future Forecasting

In [ ]:
!python scripts/forecast.py --model ensemble --use-best-params

## Cell 7 — Lihat Hasil

In [ ]:
import pandas as pd

try:
    df = pd.read_csv("results/metrics/results_summary.csv")
    print("=" * 70)
    print("REKAPITULASI HASIL")
    print("=" * 70)
    display(df.sort_values(["Scenario", "RMSE"]).reset_index(drop=True))

    print("\n" + "=" * 70)
    print("BEST MODEL PER SKENARIO")
    print("=" * 70)
    best = df.loc[df.groupby("Scenario")["RMSE"].idxmin()]
    display(best[["Scenario", "Model", "RMSE", "MAPE (%)", "MAE", "R\u00b2"]])
except FileNotFoundError:
    print("results_summary.csv belum ada.")

## Cell 8 — Tampilkan Plot

In [ ]:
import glob
from IPython.display import display, Image

plots = sorted(glob.glob("results/plots/*.png"))
print(f"{len(plots)} plots found:\n")
for p in plots:
    print(f"--- {os.path.basename(p)} ---")
    display(Image(filename=p, width=900))

## Cell 9 — Lihat Forecast

In [ ]:
import os
from IPython.display import display, Image

forecast_plots = sorted(glob.glob("results/plots/forecast_*.png"))
for p in forecast_plots:
    print(f"--- {os.path.basename(p)} ---")
    display(Image(filename=p, width=900))

# Tampilkan submission
try:
    sub = pd.read_csv("data/submissions/submission.csv")
    print(f"\nSubmission: {len(sub)} rows")
    print(sub.head())
except:
    print("Submission belum ada.")

## Cell 10 — Package Artifacts untuk Download

**Cara download hasil dari Kaggle:**
1. Jalankan cell di bawah
2. Buka menu **Output** (panel kanan bawah)
3. Klik ikon **download** di sebelah file `.tar.gz`

In [ ]:
import tarfile

# Package results + runs + submission
output_file = "/kaggle/working/experiment_artifacts.tar.gz"

with tarfile.open(output_file, "w:gz") as tar:
    for folder in ["results", "runs"]:
        if os.path.exists(folder):
            tar.add(folder)
    if os.path.exists("data/submissions/submission.csv"):
        tar.add("data/submissions/submission.csv")

print(f"Packaged: {output_file}")
print(f"Size: {os.path.getsize(output_file) / 1024:.0f} KB")
print("\nDownload dari menu Output di panel kanan bawah.")